# 01. 학습 데이터 준비

ML/DL 학습에 쓸 라벨을 모으고, **학습/시험 분할을 여기서 한 번 고정**한다.

이 노트북이 만드는 것

| 파일 | 내용 |
|---|---|
| `eval/splits.json` | 질의 60건의 학습/시험 분할. **한 번 정하면 바꾸지 않는다** |
| `ml/data/age_labels.jsonl` | 업력 근거 문장 라벨 (P2 분류기용) |

---

**실행 전 준비**

1. `data-collection/.env` 가 있어야 한다 (EC2 MySQL 접속). git 에 없으므로 직접 복사한다.
2. `pip install -r requirements-ml.txt`
3. GPU 를 쓰려면 torch 를 CUDA 판으로 따로 깐다 (아래 칸에서 확인)

In [1]:
# -*- coding: utf-8 -*-
import os, sys, json, collections, random

# 이 노트북은 data-collection/ml/ 안에 있다. 그런데 저장소의 다른 코드와 .env 는
# data-collection/ 을 기준으로 상대경로를 쓴다 (예: MYSQL_SSL_CA=data/ec2-ca.pem).
# 그래서 작업 폴더를 상위로 옮겨 놓고 시작한다. 안 그러면 DB 접속에서 실패한다.
ROOT = os.path.abspath('..')
if os.path.basename(os.getcwd()) == 'ml':
    os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print('작업 폴더 :', os.getcwd())
EVAL = os.path.join(ROOT, 'eval')
OUT  = os.path.join(ROOT, 'ml', 'data')
os.makedirs(OUT, exist_ok=True)

print('저장소 :', ROOT)
print('.env   :', '있음' if os.path.exists(os.path.join(ROOT, '.env')) else '없음 ← 복사해야 한다')

작업 폴더 : C:\SKN-TEST\SKN32-FINAL-1TEAM\data-collection
저장소 : C:\SKN-TEST\SKN32-FINAL-1TEAM\data-collection
.env   : 있음


In [2]:
# GPU 확인. False 가 나오면 torch 가 CPU 판이라 학습이 매우 느려진다.
try:
    import torch
    print('torch', torch.__version__)
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print('CUDA 사용 가능 :', name, '· VRAM %.1fGB' % vram)
    else:
        print('CUDA 사용 불가 — CPU 판이다. 아래로 다시 설치한다.')
        print('  pip uninstall -y torch')
        print('  pip install torch --index-url https://download.pytorch.org/whl/cu124')
except ImportError:
    print('torch 가 없다. P2(고전 ML)만 하려면 없어도 된다.')

torch 2.11.0+cu128
CUDA 사용 가능 : NVIDIA GeForce RTX 3060 · VRAM 12.0GB


---

## 1. 검색 관련도 라벨 — 무엇이 있나

`eval/qrels.jsonl` 이 곧 학습 데이터다. 새로 만들 필요가 없다.

- `topic_rel` **2** 딱 맞음 / **1** 분야는 걸치나 핵심 다름 / **0** 무관
- `judge` 가 `human` 이면 사람 판정, `llm_old` 면 LLM 판정

In [3]:
def read_jsonl(path):
    with io.open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

import io
queries = read_jsonl(os.path.join(EVAL, 'queries.jsonl'))
qrels   = read_jsonl(os.path.join(EVAL, 'qrels.jsonl'))

print('질의 %d건 · 판정 %d쌍' % (len(queries), len(qrels)))
print()
print('관련도 분포 :', dict(sorted(collections.Counter(r['topic_rel'] for r in qrels).items())))
print('판정자 분포 :', dict(collections.Counter(r['judge'] for r in qrels)))
print('질의 종류   :', dict(collections.Counter(q['kind'] for q in queries)))

질의 60건 · 판정 1317쌍

관련도 분포 : {0: 441, 1: 240, 2: 636}
판정자 분포 : {'llm_old': 1174, 'human': 143}
질의 종류   : {'normal': 52, 'negative': 8}


### 분야(category)는 층화 기준으로 쓸 수 없다

분야가 너무 잘게 나뉘어 있어서, 대부분 1~3건뿐이다.
1건짜리 분야는 학습/시험 어느 쪽에 넣어도 한쪽이 비게 된다.

그래서 **층화는 `kind`(정상/무관)로만 하고, 분야는 나눈 뒤 분포를 확인만 한다.**

In [4]:
cat = collections.Counter(q['category'] for q in queries)
print('분야 %d종 / 질의 %d건' % (len(cat), len(queries)))
print()
for name, n in cat.most_common():
    print('  %-14s %d' % (name, n))

분야 27종 / 질의 60건

  무관             8
  창업 일반          6
  IT·SW          4
  제조             3
  바이오·헬스         3
  농식품            3
  외식·소상공인        3
  환경·에너지         3
  유통·커머스         2
  콘텐츠            2
  모빌리티           2
  사회적경제          2
  인력             2
  금융             2
  공간·보육          2
  경영             2
  소재·화학          1
  보안             1
  수출             1
  관광             1
  디자인            1
  교육             1
  지식재산           1
  시험·인증          1
  해양·수산          1
  건설·부동산         1
  반도체·전자         1


---

## 2. 왜 질의 단위로 나눠야 하나 — 데이터 누수

「사람 판정을 시험용, LLM 판정을 학습용」으로 나누고 싶어지지만 **그러면 안 된다.**

사람 판정이 특정 질의에 몰려 있는 게 아니라 **여러 질의에 흩어져** 있기 때문이다.
같은 질의가 학습과 시험 양쪽에 걸치면, 모델이 시험 문제를 미리 본 셈이 되어
성능이 실제보다 높게 나온다. 그렇게 나온 숫자는 결과서에 쓸 수 없다.

아래에서 실제로 그런지 확인한다.

In [5]:
by_q = collections.defaultdict(collections.Counter)
for r in qrels:
    by_q[r['qid']][r['judge']] += 1

has_human = [q for q, c in by_q.items() if c['human']]
has_both  = [q for q, c in by_q.items() if c['human'] and (c['llm_old'] or c['llm'])]

print('판정이 있는 질의        : %d건' % len(by_q))
print('사람 판정이 있는 질의    : %d건' % len(has_human))
print('사람+LLM 이 섞인 질의    : %d건  ← 이만큼이 누수 위험' % len(has_both))
print()
print('판정자로 나누면 %d개 질의가 학습·시험 양쪽에 걸친다.' % len(has_both))
print('그래서 질의 단위로 나눈다.')

판정이 있는 질의        : 60건
사람 판정이 있는 질의    : 49건
사람+LLM 이 섞인 질의    : 49건  ← 이만큼이 누수 위험

판정자로 나누면 49개 질의가 학습·시험 양쪽에 걸친다.
그래서 질의 단위로 나눈다.


---

## 3. 학습/시험 분할 고정

**기준**

- 질의 단위로 나눈다 (한 질의의 판정은 전부 같은 쪽으로 간다)
- `kind` 로 층화한다 — 정상 질의와 무관 질의의 비율을 양쪽에서 맞춘다
- 학습 40 : 시험 20 (2:1)
- `SEED` 를 고정해 몇 번을 돌려도 같은 분할이 나오게 한다

⚠ **이 분할은 한 번 정하면 바꾸지 않는다.** 바꾸면 그 전에 낸 성능 수치가 전부 무의미해진다.
이미 `splits.json` 이 있으면 덮어쓰지 않고 그대로 읽는다.

In [6]:
SEED = 20260916
TEST_RATIO = 1/3          # 60건 중 20건을 시험용으로
SPLITS = os.path.join(EVAL, 'splits.json')

if os.path.exists(SPLITS):
    splits = json.load(io.open(SPLITS, encoding='utf-8'))
    print('이미 있다. 그대로 쓴다 →', SPLITS)
else:
    rng = random.Random(SEED)
    train, test = [], []
    # kind 별로 따로 섞어 같은 비율로 자른다 (층화)
    for kind in ('normal', 'negative'):
        group = sorted(q['qid'] for q in queries if q['kind'] == kind)
        rng.shuffle(group)
        cut = round(len(group) * TEST_RATIO)
        test += group[:cut]
        train += group[cut:]
    splits = {
        'seed': SEED,
        'stratify': ['kind'],
        'note': 'category 는 종류가 26개라 대부분 1~3건뿐이어서 층화 기준으로 쓰지 않았다.',
        'train': sorted(train),
        'test': sorted(test),
    }
    with io.open(SPLITS, 'w', encoding='utf-8') as f:
        json.dump(splits, f, ensure_ascii=False, indent=1)
    print('새로 만들었다 →', SPLITS)

print('학습 %d질의 · 시험 %d질의' % (len(splits['train']), len(splits['test'])))

이미 있다. 그대로 쓴다 → C:\SKN-TEST\SKN32-FINAL-1TEAM\data-collection\eval\splits.json
학습 40질의 · 시험 20질의


In [7]:
# 분할 검증 — 겹치는 질의가 없는지, 쌍이 몇 개씩 가는지, 분야가 한쪽에 쏠리지 않는지
tr, te = set(splits['train']), set(splits['test'])
assert not (tr & te), '학습과 시험에 같은 질의가 있다'
assert len(tr | te) == len(queries), '빠진 질의가 있다'
print('겹침 없음 · 빠짐 없음')
print()

kind_of = {q['qid']: q['kind'] for q in queries}
cat_of  = {q['qid']: q['category'] for q in queries}

for name, group in (('학습', tr), ('시험', te)):
    pairs = sum(1 for r in qrels if r['qid'] in group)
    human = sum(1 for r in qrels if r['qid'] in group and r['judge'] == 'human')
    kinds = collections.Counter(kind_of[q] for q in group)
    print('%s : 질의 %2d · 판정쌍 %4d (사람 %3d) · 정상 %d / 무관 %d'
          % (name, len(group), pairs, human, kinds['normal'], kinds['negative']))

print()
print('분야 쏠림 확인 (학습에만 있거나 시험에만 있는 분야)')
tr_cat = set(cat_of[q] for q in tr)
te_cat = set(cat_of[q] for q in te)
print('  시험에만 있는 분야 :', sorted(te_cat - tr_cat))
print('  → 이 분야들은 학습 때 본 적 없는 분야다. 성능이 낮게 나와도 정상이다.')

겹침 없음 · 빠짐 없음

학습 : 질의 40 · 판정쌍  893 (사람  94) · 정상 35 / 무관 5
시험 : 질의 20 · 판정쌍  424 (사람  49) · 정상 17 / 무관 3

분야 쏠림 확인 (학습에만 있거나 시험에만 있는 분야)
  시험에만 있는 분야 : ['건설·부동산', '경영', '사회적경제', '소재·화학', '유통·커머스', '콘텐츠']
  → 이 분야들은 학습 때 본 적 없는 분야다. 성능이 낮게 나와도 정상이다.


---

## 4. 업력 근거 문장 라벨 (P2 분류기용)

LLM 이 공고문에서 업력 조건을 뽑을 때 **근거 문장(`age_source_quote`)** 을 같이 남긴다.
그 값이 규칙 검산을 통과했는지로 라벨을 만든다.

```
근거 문장이 있다
  ├ age_years_max 값이 남았다   → 라벨 1 : 진짜 업력 조건
  ├ no_age_limit 표시가 있다    → 라벨 1 : 업력 제한 없음 (업력 얘기는 맞다)
  └ 값이 비었다                → 라벨 0 : 업력이 아닌 것 (사람 나이·근속·개월)
```

즉 **추가 라벨링 없이 학습 데이터가 나온다.** 다만 이건 사람이 아니라 규칙이 만든
**약한 라벨**이므로, 나중에 일부를 사람이 검수해 정확도를 확인해야 한다.

In [8]:
# EC2 MySQL 에서 근거 문장을 읽는다. .env 가 있어야 한다.
from shared import store_mysql

conn = store_mysql.connect()
cur = conn.cursor()
cur.execute('''
    SELECT notice_id, age_source_quote, age_years_max, age_years_min, no_age_limit
      FROM notice_conditions
     WHERE age_source_quote IS NOT NULL AND age_source_quote <> ''
''')
rows = cur.fetchall()
print('근거 문장이 있는 공고 %d건' % len(rows))

근거 문장이 있는 공고 332건


In [9]:
labels = []
for notice_id, quote, ymax, ymin, no_limit in rows:
    is_age = 1 if (ymax is not None or ymin is not None or no_limit) else 0
    labels.append({
        'notice_id': notice_id,
        'quote': ' '.join(str(quote).split()),   # 연속 공백 정리
        'label': is_age,                          # 1 = 업력 조건 / 0 = 업력 아님
        'age_years_max': ymax,
        'age_years_min': ymin,
        'no_age_limit': int(bool(no_limit)),
    })

path = os.path.join(OUT, 'age_labels.jsonl')
with io.open(path, 'w', encoding='utf-8') as f:
    for r in labels:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

dist = collections.Counter(r['label'] for r in labels)
print('저장 →', path)
print('라벨 1 (업력 조건)   : %d건' % dist[1])
print('라벨 0 (업력 아님)   : %d건' % dist[0])
print('합계                : %d건' % len(labels))

저장 → C:\SKN-TEST\SKN32-FINAL-1TEAM\data-collection\ml\data\age_labels.jsonl
라벨 1 (업력 조건)   : 130건
라벨 0 (업력 아님)   : 202건
합계                : 332건


In [10]:
# 실제로 어떻게 생겼는지 눈으로 본다. 라벨이 말이 되는지 확인하는 단계다.
for target, title in ((1, '라벨 1 — 진짜 업력 조건'), (0, '라벨 0 — 업력이 아닌 것')):
    print('=' * 70)
    print(title)
    print('=' * 70)
    shown = [r for r in labels if r['label'] == target][:8]
    for r in shown:
        print('  %-60s' % r['quote'][:60])
    print()

라벨 1 — 진짜 업력 조건
  업력 10년 이내 기업                                                
  관별 공고일 기준으로 창업 후 7년 이내 중소기업                                 
  청년 창업) 사업주가 만18세 ~ 39세 이하이면서, 창업 후 3년 이내인 자                 
  개업일 기준 만 3년 미만의 창업기업                                        
  청년기술창업자창업 후 년 이내로                                           
  창업 7년이상 사업장 추가 신축지원 후 지점사업자등록을 폐쇄한 경우                       
  진안군 내에서 1년 이상 사업자로 등록되어 영업 중인 소상공인, 중소기업                    
  "창업 7년 이내 소상공인"                                             

라벨 0 — 업력이 아닌 것
  ‘20.1.1.~’25.12.31. 사업자등록 신청하고 1개월 이상 계속 사업하거나              
  핵심인력 : 직무기여도가 높아 기업의 대표자가 장기재직(3년 이상)이 필요하다고 지정한 근로자        
  저신용기업 1년 이상 CC 이상 1억원 이내 창업기업 6개월 미만 - 3천만원 이내 6개월 이상~1년 미만 
  대전광역시 서구에 사업장을 두고 3개월 이상 영업 중인 소상공인                         
  도내에 사업장을 두고 3개월 이상 가동 중인 중소기업 등                             
  도내에 사업장을 두고 3개월 이상 운영 중인 건설업 중소기업으로 건설업(한국산업분류코드 41~42 중 별표1
  신청일 현재 매출액이 없더라도, 설립연도가 3년 미만인 기업은 추천가능 (일반 2억원, 우대 3억원 이내) 
  (개인은 사업자등록일, 법인은 설립등

---

## 정리

만들어진 것

- `eval/splits.json` — 학습/시험 분할. **이후 모든 실험이 이 분할을 쓴다**
- `ml/data/age_labels.jsonl` — 업력 근거 문장 라벨

다음 단계

| 노트북 | 내용 |
|---|---|
| `02_age_classifier.ipynb` | P2 — 근거 문장 분류기 (고전 ML). 반나절 |
| `03_reranker_train.py` | P1 — 리랭커 파인튜닝 (딥러닝). GPU |
| `04_reranker_report.ipynb` | P1 평가·비교표 |

**평가는 `eval/evaluate.py` 를 그대로 호출한다.** 노트북에서 지표를 새로 구현하면
기존 베이스라인(dense 0.494 / RRF 0.577)과 비교가 되지 않는다.